# Final evaluation summary

One row per setup (24 rows) with all seven evaluation metrics, collected from the three result sources. The source files are read, never changed.

| # | Metric | Source | How it is summarised per setup |
|---|---|---|---|
| 1 | Task success | `../data/prepared/baseline_run1.csv` | % of 13 cases where `answer_correctness_success_manual` is True and `ticket_status == expected_ticket_status` |
| 2 | Tool-call accuracy | `../data/prepared/baseline_run1.csv` | % of cases where `tool_correctness_success_manual` and `argument_correctness_success_manual` are both True; also the mean `tool_correctness_score` and mean `argument_correctness_score` |
| 3 | Step efficiency | `../data/prepared/baseline_run1.csv` | % of cases where `step_efficiency_trace_success_manual` is True; also the mean `step_efficiency_trace_score` |
| 4 | Cost | `../data/prepared/baseline_run1.csv` + `configs/base.yaml` pricing | Median USD per case, from input and output tokens |
| 5 | Latency | `../data/prepared/baseline_run1.csv` | Median `elapsed_s` per case |
| 6 | Robustness | `../output/tables/robustness_by_setup.csv` | % of cases passing under original, paraphrased and messy wording, for task, tool and step |
| 7 | Consistency | `../output/tables/consistency_by_setup.csv` | % of cases with identical outcomes across the three executed runs (run 1, rerun 2, rerun 3), for task, tool and step; % of cases with cost within ±10% and latency within ±20% |

**Which run.** Metrics 1–5 come from run 1, the original evaluation run, because the robustness rounds and the paired significance tests in the report are based on it. The `*_3run_pct` columns repeat metrics 1–3 averaged over all three runs (39 records per setup instead of 13), which is the more reliable basis for ranking individual variants. All three runs use `openai/gpt-oss-120b` as agent model and as judge.

Medians are used for cost and latency because both are heavily skewed: a few multi-step runs use far more tokens and time than typical lookups.

Rerun the robustness and consistency notebooks first if their inputs change, then rerun this notebook.

In [1]:
from pathlib import Path
import io
import pandas as pd
import yaml

cwd = Path.cwd()
candidates = [
    cwd,
    cwd / "results/deepeval/analysis",
    cwd / "GenAIProject/results/deepeval/analysis",
]
DIRECTORY = next((p for p in candidates if (p / "3_scorecard.ipynb").exists()), None)
if DIRECTORY is None:
    raise FileNotFoundError("Open the notebook from the project or its own directory.")

DEEPEVAL = DIRECTORY.parent
PREPARED = DEEPEVAL / "data" / "prepared"   # analysis inputs, one file per run
TABLES = DEEPEVAL / "output" / "tables"     # consistency/robustness inputs and scorecard outputs
BASELINE = PREPARED / "baseline_run1.csv"
ROBUSTNESS = TABLES / "robustness_by_setup.csv"
CONSISTENCY = TABLES / "consistency_by_setup.csv"
RERUNS = [PREPARED / f"baseline_run{i}.csv" for i in (2, 3)]
SETUPS_YAML = DEEPEVAL.parents[1] / "configs" / "setups.yaml"
BASE_YAML = DEEPEVAL.parents[1] / "configs" / "base.yaml"
CASE_IDS = [f"EVAL-{i:02d}" for i in range(1, 14)]

## 1. Setup descriptions

Each setup ID such as `s3-c` names an architecture (S3) and a reasoning-strategy variant (c), as defined in `configs/setups.yaml`.

In [2]:
config = yaml.safe_load(SETUPS_YAML.read_text())


def describe(setup_id):
    group, variant = setup_id.split("-")
    arch = config["setups"][group.upper()]
    strategy = config["variants"][arch["agents"]][variant]
    return {
        "setup": setup_id,
        "agents": arch["agents"],
        "control": arch["control"],
        "policy": arch["policy"],
        "strategy": ", ".join(f"{role}={mode}" for role, mode in strategy.items()),
    }

## 2. Metrics 1–5 from the evaluation run

Pass rates use only the `*_success_manual` judgment columns, as in the robustness and consistency notebooks. The mean scores are the judge's raw 0–1 scores. The last block averages the three pass rates over run 1 and the two reruns (built by the consistency notebook with the same columns and rules).

**Cost in USD** is derived from each run's token counts and the model price in `configs/base.yaml` (`llm.pricing`, USD per million tokens). Every agent role uses the same hosted model, so for one run:

$$
\text{cost (USD)} = \frac{\text{input tokens} \times p_{in} + \text{output tokens} \times p_{out}}{1{,}000{,}000}
$$

This is the same formula the application uses for the `cost_usd` it records in each trace. Check that the configured rate still matches the provider's current price before reporting.

In [3]:
def read_input(path):
    # Older copies of run 1 were Excel workbooks saved with a .csv name; read either format.
    content = path.read_bytes()
    if content[:2] == b"PK":
        return pd.read_excel(io.BytesIO(content))
    return pd.read_csv(path)


base = read_input(BASELINE)
base = base.loc[base.case_id.isin(CASE_IDS)].copy()
if base.duplicated(["case_id", "setup", "ticket_id"]).any():
    raise ValueError("duplicate case/setup/ticket keys in the baseline")

base["task_pass"] = base.answer_correctness_success_manual.astype(bool) & (
    base.ticket_status.astype(str) == base.expected_ticket_status.astype(str))
base["tool_pass"] = (base.tool_correctness_success_manual.astype(bool)
                     & base.argument_correctness_success_manual.astype(bool))
base["step_pass"] = base.step_efficiency_trace_success_manual.astype(bool)

llm = yaml.safe_load(BASE_YAML.read_text())["llm"]
models = set(llm["models"].values())
if len(models) != 1:
    raise ValueError(f"expected one model for every role, found {models}")
MODEL = models.pop()
PRICE = llm["pricing"][MODEL]  # USD per million tokens
base["cost_usd"] = (base.input_tokens * PRICE["input"] + base.output_tokens * PRICE["output"]) / 1_000_000
print(f"Pricing {MODEL}: ${PRICE['input']} input / ${PRICE['output']} output per million tokens")

core = base.groupby("setup").agg(
    cases=("case_id", "nunique"),
    task_success_pct=("task_pass", "mean"),
    tool_call_accuracy_pct=("tool_pass", "mean"),
    step_efficiency_pct=("step_pass", "mean"),
    tool_correctness_mean_score=("tool_correctness_score", "mean"),
    argument_correctness_mean_score=("argument_correctness_score", "mean"),
    step_efficiency_mean_score=("step_efficiency_trace_score", "mean"),
    median_cost_usd=("cost_usd", "median"),
    median_latency_s=("elapsed_s", "median"),
).reset_index()
for col in ["task_success_pct", "tool_call_accuracy_pct", "step_efficiency_pct"]:
    core[col] *= 100
assert (core.cases == len(CASE_IDS)).all(), "every setup needs all 13 cases"

# Metrics 1-3 averaged over run 1 and the two reruns
def pass_flags(frame):
    flags = frame[["case_id", "setup", "ticket_id"]].copy()
    flags["task_pass"] = frame.answer_correctness_success_manual.astype(bool) & (
        frame.ticket_status.astype(str) == frame.expected_ticket_status.astype(str))
    flags["tool_pass"] = (frame.tool_correctness_success_manual.astype(bool)
                          & frame.argument_correctness_success_manual.astype(bool))
    flags["step_pass"] = frame.step_efficiency_trace_success_manual.astype(bool)
    return flags


all_runs = pd.concat([pass_flags(base)] + [pass_flags(read_input(p).loc[lambda f: f.case_id.isin(CASE_IDS)]) for p in RERUNS])
three_runs = all_runs.groupby("setup").agg(
    task_success_3run_pct=("task_pass", "mean"),
    tool_call_accuracy_3run_pct=("tool_pass", "mean"),
    step_efficiency_3run_pct=("step_pass", "mean"),
    records_3run=("case_id", "size"),
).reset_index()
assert (three_runs.records_3run == 3 * len(CASE_IDS)).all(), "every setup needs 13 cases in each of the 3 runs"
three_runs = three_runs.drop(columns="records_3run")
three_runs[["task_success_3run_pct", "tool_call_accuracy_3run_pct", "step_efficiency_3run_pct"]] *= 100
core = core.merge(three_runs, on="setup", validate="one_to_one")
core.round(2)

Pricing openai/gpt-oss-120b: $0.03 input / $0.17 output per million tokens


,setup,cases,task_success_pct,tool_call_accuracy_pct,step_efficiency_pct,tool_correctness_mean_score,argument_correctness_mean_score,step_efficiency_mean_score,median_cost_usd,median_latency_s,task_success_3run_pct,tool_call_accuracy_3run_pct,step_efficiency_3run_pct
0,s1-a,13,76.92,84.62,84.62,0.87,0.88,0.87,0.0,24.91,89.74,87.18,89.74
1,s1-b,13,100.00,92.31,92.31,0.97,0.86,0.89,0.0,29.89,94.87,87.18,89.74
2,s2-a,13,92.31,84.62,100.00,0.89,0.86,0.99,0.0,24.93,94.87,87.18,97.44
3,s2-b,13,84.62,84.62,100.00,0.95,0.79,0.98,0.0,39.01,89.74,89.74,92.31
4,s3-a,13,76.92,84.62,100.00,0.89,0.87,0.88,0.0,37.80,82.05,87.18,94.87
5,s3-b,13,76.92,76.92,92.31,0.90,0.83,0.87,0.0,46.60,87.18,84.62,89.74
6,s3-c,13,92.31,92.31,100.00,0.92,0.90,0.90,0.0,40.19,89.74,89.74,97.44
7,s3-d,13,92.31,84.62,100.00,0.92,0.84,0.90,0.0,36.95,89.74,84.62,92.31
8,s4-a,13,84.62,76.92,92.31,0.92,0.81,0.88,0.0,34.30,84.62,79.49,89.74
9,s4-b,13,76.92,84.62,92.31,0.92,0.81,0.88,0.0,37.63,84.62,84.62,87.18


### Cost compared with a human-handled ticket

`configs/base.yaml` sets `human_cost_per_ticket_usd` as the baseline for an acceptable agent cost. Cite the source of that figure in the report.

In [4]:
HUMAN_COST = yaml.safe_load(BASE_YAML.read_text())["human_cost_per_ticket_usd"]
cost_view = core[["setup", "median_cost_usd"]].copy()
cost_view["median_cost_as_pct_of_human"] = cost_view.median_cost_usd / HUMAN_COST * 100
print(f"Human baseline: ${HUMAN_COST:.2f} per ticket; most expensive single run: ${base.cost_usd.max():.4f}")
cost_view.round(6)

Human baseline: $4.00 per ticket; most expensive single run: $0.0058


,setup,median_cost_usd,median_cost_as_pct_of_human
0,s1-a,0.000398,0.009939
1,s1-b,0.000448,0.011192
2,s2-a,0.000580,0.014508
3,s2-b,0.000501,0.012515
4,s3-a,0.000501,0.012515
5,s3-b,0.000658,0.016445
6,s3-c,0.000519,0.012968
7,s3-d,0.000522,0.013059
8,s4-a,0.000546,0.013646
9,s4-b,0.000573,0.014322


## 3. Metrics 6 and 7 from the robustness and consistency summaries

In [5]:
robustness = pd.read_csv(ROBUSTNESS)[[
    "setup", "task_success_robust_pct", "tool_call_accuracy_robust_pct", "step_efficiency_robust_pct",
]]
consistency = pd.read_csv(CONSISTENCY)[[
    "setup", "task_success_consistency_pct", "tool_call_accuracy_consistency_pct",
    "step_efficiency_consistency_pct", "cost_usd_stable_cases_pct", "elapsed_s_stable_cases_pct",
]]

summary = (
    pd.DataFrame([describe(s) for s in sorted(core.setup)])
    .merge(core.drop(columns="cases"), on="setup", how="left", validate="one_to_one")
    .merge(robustness, on="setup", how="left", validate="one_to_one")
    .merge(consistency, on="setup", how="left", validate="one_to_one")
)
missing = summary.columns[summary.isna().any()].tolist()
if missing:
    raise ValueError(f"setups missing from a source file: {missing}")
summary.round(2)

,setup,agents,control,policy,strategy,task_success_pct,tool_call_accuracy_pct,step_efficiency_pct,tool_correctness_mean_score,argument_correctness_mean_score,...,tool_call_accuracy_3run_pct,step_efficiency_3run_pct,task_success_robust_pct,tool_call_accuracy_robust_pct,step_efficiency_robust_pct,task_success_consistency_pct,tool_call_accuracy_consistency_pct,step_efficiency_consistency_pct,cost_usd_stable_cases_pct,elapsed_s_stable_cases_pct
0,s1-a,single,agentic,rag,agent=react,76.92,84.62,84.62,0.87,0.88,...,87.18,89.74,53.85,84.62,76.92,69.23,92.31,84.62,92.31,0.00
1,s1-b,single,agentic,rag,agent=plan_execute,100.00,92.31,92.31,0.97,0.86,...,87.18,89.74,92.31,84.62,92.31,84.62,84.62,76.92,23.08,23.08
2,s2-a,single,agentic,skills,agent=react,92.31,84.62,100.00,0.89,0.86,...,87.18,97.44,84.62,76.92,92.31,92.31,92.31,92.31,69.23,7.69
3,s2-b,single,agentic,skills,agent=plan_execute,84.62,84.62,100.00,0.95,0.79,...,89.74,92.31,76.92,84.62,92.31,69.23,92.31,76.92,30.77,30.77
4,s3-a,multi,agentic,rag,"supervisor=react, specialists=react",76.92,84.62,100.00,0.89,0.87,...,87.18,94.87,69.23,69.23,92.31,61.54,92.31,84.62,30.77,38.46
5,s3-b,multi,agentic,rag,"supervisor=plan_execute, specialists=plan_execute",76.92,76.92,92.31,0.90,0.83,...,84.62,89.74,76.92,61.54,92.31,61.54,76.92,69.23,15.38,23.08
6,s3-c,multi,agentic,rag,"supervisor=react, specialists=plan_execute",92.31,92.31,100.00,0.92,0.90,...,89.74,97.44,84.62,84.62,100.00,76.92,92.31,92.31,53.85,23.08
7,s3-d,multi,agentic,rag,"supervisor=plan_execute, specialists=react",92.31,84.62,100.00,0.92,0.84,...,84.62,92.31,92.31,76.92,84.62,69.23,100.00,76.92,38.46,23.08
8,s4-a,multi,agentic,skills,"supervisor=react, specialists=react",84.62,76.92,92.31,0.92,0.81,...,79.49,89.74,84.62,69.23,92.31,53.85,76.92,76.92,46.15,15.38
9,s4-b,multi,agentic,skills,"supervisor=plan_execute, specialists=plan_execute",76.92,84.62,92.31,0.92,0.81,...,84.62,87.18,76.92,76.92,84.62,84.62,84.62,92.31,15.38,30.77


## 4. Column guide

Every column in the summary, the metric it belongs to, and where its numbers come from. The `data_basis` column records where each value comes from: the executed evaluation run, the executed robustness rounds (paraphrased and messy wording), the three executed runs (run 1 and the two reruns), or the consistency comparison of those three runs.

In [6]:
MEASURED = "executed evaluation run"
ROBUST = "executed robustness rounds"
THREE_RUNS = "run 1 + rerun 2 + rerun 3"
CONSISTENT = "three executed runs compared"
guide = pd.DataFrame([
    ("setup", "-", "Setup ID (architecture + strategy variant)", "configs/setups.yaml", "-"),
    ("agents", "-", "Single agent or multi-agent", "configs/setups.yaml", "-"),
    ("control", "-", "Agentic or graph control flow", "configs/setups.yaml", "-"),
    ("policy", "-", "Policy source: RAG or skills", "configs/setups.yaml", "-"),
    ("strategy", "-", "React / plan_execute per role", "configs/setups.yaml", "-"),
    ("task_success_pct", "1 Task success", "% of cases with correct answer and matching ticket status", "data/prepared/baseline_run1.csv", MEASURED),
    ("tool_call_accuracy_pct", "2 Tool-call accuracy", "% of cases with correct tools and arguments", "data/prepared/baseline_run1.csv", MEASURED),
    ("step_efficiency_pct", "3 Step efficiency", "% of cases with no unnecessary steps", "data/prepared/baseline_run1.csv", MEASURED),
    ("tool_correctness_mean_score", "2 Tool-call accuracy", "Mean tool-correctness score (0-1)", "data/prepared/baseline_run1.csv", MEASURED),
    ("argument_correctness_mean_score", "2 Tool-call accuracy", "Mean argument-correctness score (0-1)", "data/prepared/baseline_run1.csv", MEASURED),
    ("step_efficiency_mean_score", "3 Step efficiency", "Mean step-efficiency score (0-1)", "data/prepared/baseline_run1.csv", MEASURED),
    ("median_cost_usd", "4 Cost", "Median USD per case (tokens × configs/base.yaml pricing)", "data/prepared/baseline_run1.csv + configs/base.yaml", MEASURED),
    ("median_latency_s", "5 Latency", "Median seconds per case", "data/prepared/baseline_run1.csv", MEASURED),
    ("task_success_3run_pct", "1 Task success", "Task success over all three runs (39 records)", "data/prepared/baseline_run1-3.csv", THREE_RUNS),
    ("tool_call_accuracy_3run_pct", "2 Tool-call accuracy", "Tool-call accuracy over all three runs (39 records)", "data/prepared/baseline_run1-3.csv", THREE_RUNS),
    ("step_efficiency_3run_pct", "3 Step efficiency", "Step efficiency over all three runs (39 records)", "data/prepared/baseline_run1-3.csv", THREE_RUNS),
    ("task_success_robust_pct", "6 Robustness", "% of cases passing task success under all three wordings", "output/tables/robustness_by_setup.csv", ROBUST),
    ("tool_call_accuracy_robust_pct", "6 Robustness", "% of cases passing tool-call accuracy under all three wordings", "output/tables/robustness_by_setup.csv", ROBUST),
    ("step_efficiency_robust_pct", "6 Robustness", "% of cases passing step efficiency under all three wordings", "output/tables/robustness_by_setup.csv", ROBUST),
    ("task_success_consistency_pct", "7 Consistency", "% of cases with the same task outcome in all three runs", "output/tables/consistency_by_setup.csv", CONSISTENT),
    ("tool_call_accuracy_consistency_pct", "7 Consistency", "% of cases with the same tool-call outcome in all three runs", "output/tables/consistency_by_setup.csv", CONSISTENT),
    ("step_efficiency_consistency_pct", "7 Consistency", "% of cases with the same step-efficiency outcome in all three runs", "output/tables/consistency_by_setup.csv", CONSISTENT),
    ("cost_usd_stable_cases_pct", "7 Consistency", "% of cases with cost within ±10% of the three-run mean", "output/tables/consistency_by_setup.csv", CONSISTENT),
    ("elapsed_s_stable_cases_pct", "7 Consistency", "% of cases with latency within ±20% of the three-run mean", "output/tables/consistency_by_setup.csv", CONSISTENT),
], columns=["column", "metric", "meaning", "source_file", "data_basis"])
assert list(guide.column) == list(summary.columns), "column guide is out of date"
guide

,column,metric,meaning,source_file,data_basis
0,setup,-,Setup ID (architecture + strategy variant),configs/setups.yaml,-
1,agents,-,Single agent or multi-agent,configs/setups.yaml,-
2,control,-,Agentic or graph control flow,configs/setups.yaml,-
3,policy,-,Policy source: RAG or skills,configs/setups.yaml,-
4,strategy,-,React / plan_execute per role,configs/setups.yaml,-
5,task_success_pct,1 Task success,% of cases with correct answer and matching ti...,data/prepared/baseline_run1.csv,executed evaluation run
6,tool_call_accuracy_pct,2 Tool-call accuracy,% of cases with correct tools and arguments,data/prepared/baseline_run1.csv,executed evaluation run
7,step_efficiency_pct,3 Step efficiency,% of cases with no unnecessary steps,data/prepared/baseline_run1.csv,executed evaluation run
8,tool_correctness_mean_score,2 Tool-call accuracy,Mean tool-correctness score (0-1),data/prepared/baseline_run1.csv,executed evaluation run
9,argument_correctness_mean_score,2 Tool-call accuracy,Mean argument-correctness score (0-1),data/prepared/baseline_run1.csv,executed evaluation run


## 5. All metrics by case category

The 13 cases fall into three categories (see `deepeval/cases/categories.csv`):

| Category | Cases | Records per setup |
|---|---|---|
| Simple lookup | EVAL-01 to EVAL-08 | 8 |
| Multi-step | EVAL-09, EVAL-10, EVAL-12, EVAL-13 | 4 |
| Safety and guardrail | EVAL-11 | 1 |

Per setup, these groups are too small to compare: a single safety case can only score 0% or 100%. Categories are therefore reported **pooled across setups**: first across all 24 setups, then split by each architecture dimension (single vs multi agent, agentic vs graph control, RAG vs skills policy), with 12 setups on each side. `records` gives the number of case/setup runs behind each row.

Robustness and consistency are grouped the same way from their per-case results (`robustness_by_setup_and_case.csv`, `consistency_by_setup_and_case.csv`): the share of case/setup records that were robust, or consistent, within each group. Robustness comes from the executed paraphrase and messy-input rounds; consistency compares run 1 with the two reruns.

In [7]:
KEY = ["case_id", "setup", "ticket_id"]
robust_cases = pd.read_csv(TABLES / "robustness_by_setup_and_case.csv")[
    KEY + ["task_robust_pass", "tool_robust_pass", "step_robust_pass"]]
consistency_cases = pd.read_csv(TABLES / "consistency_by_setup_and_case.csv")[
    KEY + ["task_same_outcome", "tool_same_outcome", "step_same_outcome",
           "cost_usd_within_margin", "elapsed_s_within_margin"]]

runs = (
    base.merge(pd.DataFrame([describe(s) for s in sorted(base.setup.unique())]), on="setup", how="left")
    .merge(robust_cases, on=KEY, how="left", validate="one_to_one")
    .merge(consistency_cases, on=KEY, how="left", validate="one_to_one")
)
if runs.isna().any().any():
    raise ValueError("a case/setup record is missing from the robustness or consistency results")
CATEGORY_ORDER = ["simple lookup", "multi-step", "safety-and-guardrail"]


def by_category(frame, dimension=None):
    keys = ["category"] if dimension is None else [dimension, "category"]
    table = frame.groupby(keys).agg(
        records=("case_id", "size"),
        task_success_pct=("task_pass", "mean"),
        tool_call_accuracy_pct=("tool_pass", "mean"),
        step_efficiency_pct=("step_pass", "mean"),
        tool_correctness_mean_score=("tool_correctness_score", "mean"),
        argument_correctness_mean_score=("argument_correctness_score", "mean"),
        step_efficiency_mean_score=("step_efficiency_trace_score", "mean"),
        median_cost_usd=("cost_usd", "median"),
        median_latency_s=("elapsed_s", "median"),
        task_success_robust_pct=("task_robust_pass", "mean"),
        tool_call_accuracy_robust_pct=("tool_robust_pass", "mean"),
        step_efficiency_robust_pct=("step_robust_pass", "mean"),
        task_success_consistency_pct=("task_same_outcome", "mean"),
        tool_call_accuracy_consistency_pct=("tool_same_outcome", "mean"),
        step_efficiency_consistency_pct=("step_same_outcome", "mean"),
        cost_usd_stable_cases_pct=("cost_usd_within_margin", "mean"),
        elapsed_s_stable_cases_pct=("elapsed_s_within_margin", "mean"),
    ).reset_index()
    for col in table.columns:
        if col.endswith("_pct"):
            table[col] = table[col].astype(float) * 100
    table.insert(0, "group_by", dimension or "all setups")
    table.insert(1, "group", table.pop(dimension) if dimension else "all")
    return table


category_summary = pd.concat(
    [by_category(runs)] + [by_category(runs, d) for d in ["agents", "control", "policy"]],
    ignore_index=True,
)
category_summary["category"] = pd.Categorical(category_summary.category, CATEGORY_ORDER, ordered=True)
category_summary = category_summary.sort_values(["group_by", "category", "group"], key=lambda s: s if s.name != "group_by" else s.map(
    {"all setups": 0, "agents": 1, "control": 2, "policy": 3})).reset_index(drop=True)
category_summary["category"] = category_summary.category.astype(str)
category_summary.round(2)

,group_by,group,category,records,task_success_pct,tool_call_accuracy_pct,step_efficiency_pct,tool_correctness_mean_score,argument_correctness_mean_score,step_efficiency_mean_score,median_cost_usd,median_latency_s,task_success_robust_pct,tool_call_accuracy_robust_pct,step_efficiency_robust_pct,task_success_consistency_pct,tool_call_accuracy_consistency_pct,step_efficiency_consistency_pct,cost_usd_stable_cases_pct,elapsed_s_stable_cases_pct
0,all setups,all,simple lookup,192,94.27,86.98,99.48,1.00,0.82,0.96,0.0,31.07,90.10,81.25,95.83,86.46,97.40,94.27,61.98,22.92
1,all setups,all,multi-step,96,66.67,71.88,89.58,0.71,0.80,0.81,0.0,56.81,59.38,58.33,86.46,63.54,59.38,60.42,17.71,29.17
2,all setups,all,safety-and-guardrail,24,100.00,100.00,91.67,1.00,1.00,0.90,0.0,1.92,95.83,100.00,79.17,100.00,100.00,83.33,45.83,4.17
3,agents,multi,simple lookup,128,95.31,87.50,99.22,1.00,0.82,0.95,0.0,33.29,90.62,81.25,96.88,85.16,96.88,92.19,58.59,25.00
4,agents,single,simple lookup,64,92.19,85.94,100.00,1.00,0.82,0.99,0.0,25.35,89.06,81.25,93.75,89.06,98.44,98.44,68.75,18.75
5,agents,multi,multi-step,64,62.50,67.19,90.62,0.70,0.81,0.81,0.0,60.77,59.38,50.00,87.50,65.62,56.25,62.50,17.19,28.12
6,agents,single,multi-step,32,75.00,81.25,87.50,0.74,0.79,0.82,0.0,48.97,59.38,75.00,84.38,59.38,65.62,56.25,18.75,31.25
7,agents,multi,safety-and-guardrail,16,100.00,100.00,100.00,1.00,1.00,0.97,0.0,1.92,100.00,100.00,87.50,100.00,100.00,93.75,50.00,0.00
8,agents,single,safety-and-guardrail,8,100.00,100.00,75.00,1.00,1.00,0.76,0.0,1.89,87.50,100.00,62.50,100.00,100.00,62.50,37.50,12.50
9,control,agentic,simple lookup,96,94.79,87.50,98.96,1.00,0.84,0.93,0.0,33.65,93.75,80.21,93.75,82.29,97.92,91.67,54.17,17.71


## 6. Save the results

| Output | Contents |
|---|---|
| `scorecard_by_setup.csv` | One row per setup with all seven metrics |
| `scorecard_columns.csv` | What each summary column means and which file it comes from |
| `scorecard_by_category.csv` | All metrics per case category, pooled across all setups and per architecture dimension |

In [8]:
summary.to_csv(TABLES / "scorecard_by_setup.csv", index=False)
guide.to_csv(TABLES / "scorecard_columns.csv", index=False)
category_summary.to_csv(TABLES / "scorecard_by_category.csv", index=False)
print("Saved the summary, column guide and category breakdown.")

Saved the summary, column guide and category breakdown.
